In [2]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix



df = pd.read_csv("IMDB Dataset.csv")

df.head()
df['sentiment'].value_counts()


def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df['clean_review'] = df['review'].apply(clean_text)

df['label'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})


X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'],
    df['label'],
    test_size=0.2,
    random_state=42
)

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)


accuracy_score(y_test, y_pred_nb)

f1_score(y_test, y_pred_nb)

print(classification_report(y_test, y_pred_nb))


cm = confusion_matrix(y_test, y_pred_nb)

print(cm)

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_tfidf, y_train)

y_pred_lr = lr_model.predict(X_test_tfidf)

lr_accuracy = accuracy_score(y_test, y_pred_lr)

lr_f1 = f1_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", lr_accuracy)

print("Logistic Regression F1:", lr_f1)

print(classification_report(y_test, y_pred_lr))

cm_lr = confusion_matrix(y_test, y_pred_lr)

print(cm_lr)


results = pd.DataFrame({
    'Model': ['Naive Bayes', 'Logistic Regression'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_lr)
    ],
    'F1 Score': [
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_lr)
    ]
})

print(results)

              precision    recall  f1-score   support

           0       0.85      0.85      0.85      4961
           1       0.85      0.85      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

[[4217  744]
 [ 770 4269]]
Logistic Regression Accuracy: 0.8934
Logistic Regression F1: 0.8955106841795727
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.88      0.91      0.90      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

[[4366  595]
 [ 471 4568]]
                 Model  Accuracy  F1 Score
0          Naive Bayes    0.8486  0.849383
1  Logistic Regression    0.8934  0.895511


In [3]:
%pip install transformers datasets torch -q

^C
Traceback (most recent call last):
  File "/Users/satyamvaishnav/anaconda3/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/satyamvaishnav/anaconda3/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/satyamvaishnav/anaconda3/lib/python3.10/site-packages/pip/__main__.py", line 31, in <module>
    sys.exit(_main())
  File "/Users/satyamvaishnav/anaconda3/lib/python3.10/site-packages/pip/_internal/cli/main.py", line 70, in main
    return command.main(cmd_args)
  File "/Users/satyamvaishnav/anaconda3/lib/python3.10/site-packages/pip/_internal/cli/base_command.py", line 101, in main
    return self._main(args)
  File "/Users/satyamvaishnav/anaconda3/lib/python3.10/site-packages/pip/_internal/cli/base_command.py", line 216, in _main
    self.handle_pip_version_check(options)
  File "/Users/satyamvaishnav/anaconda3/lib/python3.10/site-packages/pip/_internal/cli/req_command.py", 

In [5]:
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

/Users/satyamvaishnav/anaconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
tokenizer = BertTokenizer.from_pretrained(
    'bert-base-uncased'
)


/Users/satyamvaishnav/anaconda3/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True
)

In [8]:
import torch

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item['labels'] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [9]:
train_dataset = IMDbDataset(
    train_encodings,
    y_train
)

test_dataset = IMDbDataset(
    test_encodings,
    y_test
)

In [10]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at

In [11]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',
    logging_steps=100
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

In [ ]:
X_train_small = X_train[:5000]
y_train_small = y_train[:5000]

In [14]:
train_encodings = tokenizer(
    X_train_small.tolist(),
    truncation=True,
    padding=True
)

In [15]:
train_dataset = IMDbDataset(
    train_encodings,
    y_train_small
)

In [16]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

/Users/satyamvaishnav/anaconda3/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
loading configuration file config.json from cache at /Users/satyamvaishnav/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absol

In [17]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',
    logging_steps=100
)

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [19]:
trainer.train()

/Users/satyamvaishnav/anaconda3/lib/python3.10/site-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 5000
  Num Epochs = 1
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 1
  Total optimization steps = 625
  Number of trainable parameters = 109483778
 16%|█▌        | 100/625 [42:25<3:26:06, 23.56s/it]

{'loss': 0.5964, 'learning_rate': 4.2e-05, 'epoch': 0.16}


 32%|███▏      | 200/625 [1:24:07<2:43:19, 23.06s/it]

{'loss': 0.4187, 'learning_rate': 3.4000000000000007e-05, 'epoch': 0.32}


 48%|████▊     | 300/625 [2:04:47<2:03:32, 22.81s/it]

...

Model weights saved in ./results/checkpoint-500/pytorch_model.bin
 96%|█████████▌| 600/625 [4:04:30<10:27, 25.11s/it]

{'loss': 0.3106, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.96}


100%|██████████| 625/625 [4:15:00<00:00, 24.29s/it]

Training completed. Do not forget to share your model on huggingface.co/models =)


100%|██████████| 625/625 [4:15:00<00:00, 24.48s/it]

{'train_runtime': 15301.0305, 'train_samples_per_second': 0.327, 'train_steps_per_second': 0.041, 'train_loss': 0.39120165252685546, 'epoch': 1.0}


TrainOutput(global_step=625, training_loss=0.39120165252685546, metrics={'train_runtime': 15301.0305, 'train_samples_per_second': 0.327, 'train_steps_per_second': 0.041, 'train_loss': 0.39120165252685546, 'epoch': 1.0})

In [20]:
predictions = trainer.predict(test_dataset)

***** Running Prediction *****
  Num examples = 10000
  Batch size = 8
100%|██████████| 1250/1250 [2:19:59<00:00,  6.72s/it] 


In [21]:
import numpy as np

y_pred_bert = np.argmax(
    predictions.predictions,
    axis=1
)

In [22]:
bert_accuracy = accuracy_score(
    y_test,
    y_pred_bert
)

bert_f1 = f1_score(
    y_test,
    y_pred_bert
)

print(bert_accuracy)
print(bert_f1)

0.903
0.9058435255290236


In [23]:
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification
)

In [24]:
distil_tokenizer = DistilBertTokenizer.from_pretrained(
    'distilbert-base-uncased'
)

/Users/satyamvaishnav/anaconda3/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
loading file vocab.txt from cache at /Users/satyamvaishnav/.cache/huggingface/hub/models--distilbert-base-uncased/snapshots/12040accade4e8a0f71eabdb258fecc2e7e948be/vocab.txt
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /Users/satyamvaishnav/.cache/huggingface/hub/models--distilbert-base-uncased/snapshots/12040accade4e8a0f71eabdb258fecc2e7e948be/tokenizer_config.json
loading configuration file config.json from cache at /Users/satyamvaishnav/.cache/huggingface/hub/models--distilbert-base-uncased/snapshots/12040accade4e8a0f71eabdb258fecc2e7e948be/config.json
Model config DistilBert

In [25]:
distil_train_encodings = distil_tokenizer(
    X_train_small.tolist(),
    truncation=True,
    padding=True
)

distil_test_encodings = distil_tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True
)

In [26]:
distil_train_dataset = IMDbDataset(
    distil_train_encodings,
    y_train_small
)

distil_test_dataset = IMDbDataset(
    distil_test_encodings,
    y_test
)

In [27]:
distil_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

loading configuration file config.json from cache at /Users/satyamvaishnav/.cache/huggingface/hub/models--distilbert-base-uncased/snapshots/12040accade4e8a0f71eabdb258fecc2e7e948be/config.json
Model config DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "hidden_dim": 3072,
  "initializer_range": 0.02,
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "transformers_version": "4.24.0",
  "vocab_size": 30522
}

loading weights file pytorch_model.bin from cache at /Users/satyamvaishnav/.cache/huggingface/hub/models--distilbert-base-uncased/snapshots/12040accade4e8a0f71eabdb258fecc2e7e948be/pytorch_model.bin
Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing Dis

In [28]:
distil_training_args = TrainingArguments(
    output_dir='./distil_results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./distil_logs',
    logging_steps=100
)

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


In [29]:
distil_trainer = Trainer(
    model=distil_model,
    args=distil_training_args,
    train_dataset=distil_train_dataset,
    eval_dataset=distil_test_dataset
)

In [ ]:
distil_trainer.train()